# TRACE: inspect the trained model

Compares three things on the **unseen test map**:
- 🔴 **model prediction**: from the camera image only (soft-argmax of each heatmap)
- 🟢 **actual path driven**: the training label (where the autopilot actually went)
- 🔵 **map centerline (hard-coded)**: the reference line the autopilot was following, traced from the map

Each row shows: **(1)** the heatmaps over the image, **(2)** all three paths in the camera view, **(3)** a top-down view in meters with the duct walls.

Needs `trace_data.tar.gz` and `trace_runs/trace_vit_heatmap.pt` in your Google Drive. A GPU is optional here; a T4 just makes the metrics cell faster.

In [ ]:
# 1. Setup: Drive, data, sim repo (for the map files)
!pip -q install timm
from google.colab import drive
drive.mount('/content/drive')
TAR  = '/content/drive/MyDrive/trace_data.tar.gz'
CKPT = '/content/drive/MyDrive/trace_runs/trace_vit_heatmap.pt'
OUT  = '/content/drive/MyDrive/trace_runs'
import os
%cd /content
if not os.path.exists('/content/data'):
    !tar xzf "$TAR" -C /content
if not os.path.exists('/content/sim'):
    !git clone -q --depth 1 https://github.com/jfasate/f1tenth_gym_gazebo.git /content/sim
MAPS_DIR = '/content/sim/maps'
!ls /content/data

In [ ]:
%%writefile /content/trace_core.py
# TRACE - ViT + heatmap trajectory model: dataset, model, loss, train/eval, viz.
# Used by the Colab notebook (each section is pasted into a cell there).
import glob, json, math, os, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image

IMG = 224          # model input (square)
HM = 56            # heatmap size (IMG / 4)

# ------------------------------------------------------------------ data
def load_split(root, maps):
    """All samples from data/<map>/labels.jsonl for the given maps."""
    out = []
    for m in maps:
        p = os.path.join(root, m, "labels.jsonl")
        with open(p) as fh:
            for l in fh:
                s = json.loads(l)
                s["img"] = os.path.join(os.path.dirname(root), s["img"]) if not os.path.isabs(s["img"]) else s["img"]
                s["map"] = m
                out.append(s)
    return out


def episode_of(s):
    return os.path.basename(os.path.dirname(os.path.dirname(s["img"])))


def split_train_val(samples, val_frac=0.15):
    """Val = the last 15% of every episode (time-contiguous, so no near-duplicate leak)."""
    by_ep = {}
    for s in samples:
        by_ep.setdefault((s["map"], episode_of(s)), []).append(s)
    tr, va = [], []
    for k, ss in by_ep.items():
        ss.sort(key=lambda s: s["img"])
        n = int(len(ss) * (1 - val_frac))
        tr += ss[:n]
        va += ss[n:]
    return tr, va


class TrajDataset(torch.utils.data.Dataset):
    """Images pre-loaded into RAM at IMG x IMG (uint8) for speed; targets in IMG pixel coords."""
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __init__(self, samples, train=False):
        self.s, self.train = samples, train
        self.imgs, self.uv, self.valid = [], [], []
        for i, s in enumerate(samples):
            im = Image.open(s["img"]).convert("RGB")
            W, H = im.size
            self.imgs.append(np.asarray(im.resize((IMG, IMG), Image.BILINEAR)))
            uv = np.array(s["uv"], dtype=np.float32)
            uv[:, 0] *= IMG / W
            uv[:, 1] *= IMG / H
            self.uv.append(uv)
            self.valid.append(np.array(s["valid"], dtype=bool))
            if (i + 1) % 2000 == 0:
                print(f"  loaded {i + 1}/{len(samples)}")
        self.imgs = np.stack(self.imgs)

    def __len__(self):
        return len(self.s)

    def __getitem__(self, i):
        x = torch.from_numpy(self.imgs[i]).permute(2, 0, 1).float() / 255.0
        uv = torch.from_numpy(self.uv[i].copy())
        valid = torch.from_numpy(self.valid[i].copy())
        if self.train:
            if random.random() < 0.5:                         # mirror image + path
                x = x.flip(-1)
                uv[:, 0] = IMG - 1 - uv[:, 0]
            b = 1 + (random.random() - 0.5) * 0.5             # brightness / contrast jitter
            c = 1 + (random.random() - 0.5) * 0.5
            m = x.mean()
            x = ((x - m) * c + m) * b
            x = x.clamp(0, 1)
        x = (x - self.MEAN) / self.STD
        return x, uv, valid


# ------------------------------------------------------------------ model
class ViTHeatmap(nn.Module):
    """Pretrained ViT encoder -> patch tokens as a 14x14 feature map -> upsampling conv
    decoder -> one 56x56 heatmap per waypoint."""

    def __init__(self, n_wp=6, backbone="vit_small_patch16_224.augreg_in21k_ft_in1k", pretrained=True):
        super().__init__()
        import timm
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        d = self.enc.num_features
        self.n_prefix = getattr(self.enc, "num_prefix_tokens", 1)
        self.grid = IMG // self.enc.patch_embed.patch_size[0]

        def up(cin, cout):
            return nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                                 nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.GELU())
        self.dec = nn.Sequential(
            nn.Conv2d(d, 256, 1), nn.BatchNorm2d(256), nn.GELU(),
            up(256, 128),                                        # 14 -> 28
            up(128, 64),                                         # 28 -> 56
            nn.Conv2d(64, 64, 3, padding=1), nn.GELU(),
            nn.Conv2d(64, n_wp, 1))

    def forward(self, x):
        t = self.enc.forward_features(x)[:, self.n_prefix:]      # (B, N, d) patch tokens
        B, N, d = t.shape
        f = t.transpose(1, 2).reshape(B, d, self.grid, self.grid)
        hm = self.dec(f)
        if hm.shape[-1] != HM:
            hm = F.interpolate(hm, size=(HM, HM), mode="bilinear", align_corners=False)
        return hm                                                 # (B, n_wp, HM, HM) logits


# ------------------------------------------------------------------ loss / decoding
_ys, _xs = torch.meshgrid(torch.arange(HM).float(), torch.arange(HM).float(), indexing="ij")


def to_hm(uv):
    """IMG pixel coords -> heatmap cell coords (cell centers)."""
    return (uv + 0.5) * (HM / IMG) - 0.5


def from_hm(c):
    return (c + 0.5) * (IMG / HM) - 0.5


def soft_argmax(logits):
    B, K, H, W = logits.shape
    p = logits.flatten(2).softmax(-1).view(B, K, H, W)
    xs, ys = _xs.to(logits.device), _ys.to(logits.device)
    return torch.stack([(p * xs).sum((-1, -2)), (p * ys).sum((-1, -2))], -1)   # heatmap coords


def loss_fn(logits, uv, valid, sigma=1.5, w_coord=0.1):
    """Cross-entropy between softmax(heatmap) and a Gaussian at the target, plus L1 on soft-argmax.
    Waypoints outside the image are ignored."""
    c = to_hm(uv)                                                   # (B, K, 2)
    xs, ys = _xs.to(uv.device), _ys.to(uv.device)
    g = torch.exp(-((xs - c[..., 0, None, None]) ** 2 + (ys - c[..., 1, None, None]) ** 2) / (2 * sigma ** 2))
    g = g / g.flatten(2).sum(-1).clamp_min(1e-8)[..., None, None]
    logp = logits.flatten(2).log_softmax(-1).view_as(logits)
    ce = -(g * logp).sum((-1, -2))                                  # (B, K)
    l1 = (soft_argmax(logits) - c).abs().sum(-1)                    # (B, K)
    m = valid.float()
    n = m.sum().clamp_min(1)
    return ((ce + w_coord * l1) * m).sum() / n


# ------------------------------------------------------------------ train / eval
def evaluate(model, loader, device, native_wh=(320, 240)):
    """Mean pixel error per waypoint (in the ORIGINAL camera resolution), visible points only."""
    model.eval()
    sx, sy = native_wh[0] / IMG, native_wh[1] / IMG
    err_sum, cnt = None, None
    with torch.no_grad():
        for x, uv, valid in loader:
            x, uv, valid = x.to(device), uv.to(device), valid.to(device)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(x)
            pred = from_hm(soft_argmax(logits.float()))
            d = pred - uv
            e = torch.sqrt((d[..., 0] * sx) ** 2 + (d[..., 1] * sy) ** 2) * valid
            err_sum = e.sum(0) if err_sum is None else err_sum + e.sum(0)
            cnt = valid.sum(0) if cnt is None else cnt + valid.sum(0)
    per_wp = (err_sum / cnt.clamp_min(1)).cpu().numpy()
    return {"ADE_px": float(per_wp.mean()), "FDE_px": float(per_wp[-1]), "per_wp_px": per_wp.round(1).tolist()}


def train(model, tr_loader, va_loader, device, epochs=15, lr_head=1e-3, lr_enc=3e-5,
          freeze_epochs=2, ckpt="trace_vit_heatmap.pt", native_wh=(320, 240)):
    enc_p = list(model.enc.parameters())
    head_p = list(model.dec.parameters())
    opt = torch.optim.AdamW([{"params": enc_p, "lr": lr_enc}, {"params": head_p, "lr": lr_head}], weight_decay=0.05)
    steps = epochs * len(tr_loader)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr_enc, lr_head], total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
    best = 1e9
    hist = []
    for ep in range(epochs):
        model.train()
        frozen = ep < freeze_epochs                                  # warm up the decoder first
        for p in enc_p:
            p.requires_grad_(not frozen)
        t0, tot, nb = time.time(), 0.0, 0
        for x, uv, valid in tr_loader:
            x, uv, valid = x.to(device), uv.to(device), valid.to(device)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(x)
            loss = loss_fn(logits.float(), uv, valid)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item()
            nb += 1
        m = evaluate(model, va_loader, device, native_wh)
        hist.append({"epoch": ep + 1, "train_loss": tot / nb, **m})
        flag = ""
        if m["ADE_px"] < best:
            best = m["ADE_px"]
            torch.save(model.state_dict(), ckpt)
            flag = "  (saved)"
        print(f"epoch {ep + 1:2d}/{epochs}  loss {tot / nb:.3f}  val ADE {m['ADE_px']:.1f}px  "
              f"FDE {m['FDE_px']:.1f}px  {'[enc frozen] ' if frozen else ''}{time.time() - t0:.0f}s{flag}")
    return hist


# ------------------------------------------------------------------ viz
def show_predictions(model, ds, device, n=12, cols=4, seed=0, save=None):
    import matplotlib.pyplot as plt
    model.eval()
    rng = random.Random(seed)
    idx = rng.sample(range(len(ds)), min(n, len(ds)))
    rows = math.ceil(len(idx) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, i in zip(axes, idx):
        x, uv, valid = ds[i]
        with torch.no_grad():
            pred = from_hm(soft_argmax(model(x[None].to(device)).float()))[0].cpu()
        ax.imshow(ds.imgs[i])
        v = valid.numpy()
        ax.plot(uv[v, 0], uv[v, 1], "o-", color="lime", lw=2, ms=4, label="ground truth")
        ax.plot(pred[:, 0], pred[:, 1], "o-", color="red", lw=2, ms=4, label="predicted")
        ax.set_xlim(0, IMG); ax.set_ylim(IMG, 0); ax.axis("off")
    for ax in axes[len(idx):]:
        ax.axis("off")
    axes[0].legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=110)
    plt.show()


In [ ]:
%%writefile /content/trace_ref.py
# TRACE - compare model output against the ground-truth path and the hard-coded map centerline.
# Needs: data/<map>/ep_*/{meta.json,frames.jsonl} (from gz_collect.py) and the repo's maps/<map>.yaml.
import json, math, os
import numpy as np
import torch
import torch.nn.functional as F

import trace_core as tc

# ------------------------------------------------------------ map + centerline (same as autodrive.py)
def load_map(yaml_path):
    import yaml
    from PIL import Image
    m = yaml.safe_load(open(yaml_path))
    g = np.array(Image.open(os.path.join(os.path.dirname(yaml_path), m["image"])).convert("L")).astype(float) / 255.0
    occ_p = g if m.get("negate", 0) else 1.0 - g
    return occ_p > m.get("occupied_thresh", 0.65), float(m["resolution"]), m["origin"]


class Field:
    def __init__(self, occ, res, origin):
        from scipy import ndimage
        self.occ = occ
        self.d = ndimage.distance_transform_edt(~occ) * res
        self.res, self.ox, self.oy, self.H = res, origin[0], origin[1], occ.shape[0]

    def __call__(self, x, y):
        c = np.clip((np.asarray(x) - self.ox) / self.res, 0, self.d.shape[1] - 1.001)
        r = np.clip((self.H - 1) - (np.asarray(y) - self.oy) / self.res, 0, self.d.shape[0] - 1.001)
        c0, r0 = np.floor(c).astype(int), np.floor(r).astype(int)
        fc, fr = c - c0, r - r0
        d = self.d
        return ((1 - fr) * ((1 - fc) * d[r0, c0] + fc * d[r0, c0 + 1])
                + fr * ((1 - fc) * d[r0 + 1, c0] + fc * d[r0 + 1, c0 + 1]))

    def walls_near(self, x, y, radius=5.0, max_pts=4000):
        """World coords of occupied cells within radius (for the bird's-eye plot)."""
        rr, cc = np.nonzero(self.occ)
        wx = self.ox + cc * self.res
        wy = self.oy + (self.H - 1 - rr) * self.res
        m = (np.abs(wx - x) < radius) & (np.abs(wy - y) < radius)
        wx, wy = wx[m], wy[m]
        if len(wx) > max_pts:
            k = np.random.default_rng(0).choice(len(wx), max_pts, replace=False)
            wx, wy = wx[k], wy[k]
        return wx, wy


def trace_centerline(field, x0, y0, yaw0, step=0.05, look=0.35, max_len=400.0):
    hs = np.linspace(-0.6, 0.6, 31)
    x, y, h = x0, y0, yaw0
    pts, travelled = [(x, y)], 0.0
    for _ in range(int(max_len / step)):
        cand = h + hs
        score = field(x + look * np.cos(cand), y + look * np.sin(cand)) - 0.03 * np.abs(hs)
        h = cand[int(np.argmax(score))]
        x, y = x + step * math.cos(h), y + step * math.sin(h)
        pts.append((x, y))
        travelled += step
        if travelled > 3.0 and math.hypot(x - x0, y - y0) < 2 * step + 0.05:
            break
    else:
        raise RuntimeError("centerline did not close into a loop")
    P = np.array(pts[:-1])
    k = 9
    pad = np.vstack([P[-k:], P, P[:k]])
    return np.stack([np.convolve(pad[:, i], np.ones(2 * k + 1) / (2 * k + 1), "valid") for i in range(2)], 1)


# ------------------------------------------------------------ per-episode context
class Episodes:
    """Looks up, for any labels.jsonl sample: the car pose at that frame, camera meta,
    and the map centerline (traced once per episode in the direction it was driven)."""

    def __init__(self, maps_dir):
        self.maps_dir = maps_dir
        self.meta, self.frames, self.center, self.fields = {}, {}, {}, {}

    def _ep(self, s):
        ep_dir = os.path.dirname(os.path.dirname(s["img"]))
        if ep_dir not in self.meta:
            self.meta[ep_dir] = json.load(open(os.path.join(ep_dir, "meta.json")))
            fr = [json.loads(l) for l in open(os.path.join(ep_dir, "frames.jsonl")) if l.strip()]
            self.frames[ep_dir] = {f["i"]: f for f in fr}
            mp = s["map"]
            if mp not in self.fields:
                self.fields[mp] = Field(*load_map(os.path.join(self.maps_dir, mp + ".yaml")))
            e0 = fr[0]["ego"]
            self.center[ep_dir] = trace_centerline(self.fields[mp], e0["x"], e0["y"], e0["yaw"])
        return ep_dir

    def context(self, s):
        ep = self._ep(s)
        i = int(os.path.splitext(os.path.basename(s["img"]))[0])
        return {"meta": self.meta[ep], "pose": self.frames[ep][i]["ego"],
                "center": self.center[ep], "field": self.fields[s["map"]]}


# ------------------------------------------------------------ geometry
def world_to_car(pts_xy, pose):
    c, s = math.cos(pose["yaw"]), math.sin(pose["yaw"])
    d = np.asarray(pts_xy) - [pose["x"], pose["y"]]
    return np.stack([c * d[:, 0] + s * d[:, 1], -s * d[:, 0] + c * d[:, 1]], 1)     # x fwd, y left


def car_to_pixels(car_xy, pose, meta):
    """Ground points in car frame -> native-resolution pixels (+ visibility)."""
    K = np.asarray(meta["K"])
    T_cam_base = np.linalg.inv(np.asarray(meta["T_base_cam"]))
    n = len(car_xy)
    pb = np.stack([car_xy[:, 0], car_xy[:, 1], np.full(n, -pose.get("z", 0.0)), np.ones(n)])
    pc = (T_cam_base @ pb)[:3]
    uvw = K @ pc
    uv = (uvw[:2] / np.clip(uvw[2], 1e-6, None)).T
    W, H = meta["width"], meta["height"]
    vis = (pc[2] > 0.05) & (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)
    return uv, vis


def pixels_to_car(uv_native, pose, meta):
    """Native pixels -> where that ray hits the floor, in car frame (meters). NaN above the horizon."""
    K_inv = np.linalg.inv(np.asarray(meta["K"]))
    T = np.asarray(meta["T_base_cam"])
    R, c = T[:3, :3], T[:3, 3]
    uv1 = np.c_[uv_native, np.ones(len(uv_native))]
    d = (R @ (K_inv @ uv1.T)).T                                   # ray directions in base frame
    z_ground = -pose.get("z", 0.0)
    t = (z_ground - c[2]) / np.where(np.abs(d[:, 2]) < 1e-9, -1e-9, d[:, 2])
    p = c[None, :] + t[:, None] * d
    p[(t <= 0) | (d[:, 2] >= 0)] = np.nan
    p[np.linalg.norm(p[:, :2], axis=1) > 10.0] = np.nan          # near-horizon pixels -> unreliable range
    return p[:, :2]


def centerline_ahead(ctx, start=0.3, end=4.0):
    """Centerline segment from `start` to `end` meters ahead of the car (along the loop), car frame."""
    P = ctx["center"]
    pose = ctx["pose"]
    j = int(np.argmin(np.linalg.norm(P - [pose["x"], pose["y"]], axis=1)))
    seg = np.linalg.norm(np.diff(np.vstack([P, P[:1]]), axis=0), axis=1)
    order = (j + np.arange(len(P))) % len(P)
    arc = np.concatenate([[0], np.cumsum(seg[order][:-1])])
    keep = (arc >= start) & (arc <= end)
    return world_to_car(P[order][keep], pose)


def dist_to_polyline(pts, line):
    """Distance from each point to the nearest segment of a polyline."""
    out = np.full(len(pts), np.nan)
    if len(line) < 2:
        return out
    a, b = line[:-1], line[1:]
    ab = b - a
    L2 = (ab ** 2).sum(1).clip(1e-12)
    for k, p in enumerate(pts):
        if np.any(np.isnan(p)):
            continue
        t = (((p - a) * ab).sum(1) / L2).clip(0, 1)
        out[k] = np.sqrt((((a + t[:, None] * ab) - p) ** 2).sum(1)).min()
    return out


# ------------------------------------------------------------ model helpers
@torch.no_grad()
def predict(model, ds, idx, device, bs=64):
    """Returns pred uv in IMG (224) coords and softmaxed heatmaps (n, K, HM, HM)."""
    model.eval()
    uvs, hms = [], []
    for b in range(0, len(idx), bs):
        x = torch.stack([ds[i][0] for i in idx[b:b + bs]]).to(device)
        logits = model(x).float()
        uvs.append(tc.from_hm(tc.soft_argmax(logits)).cpu())
        B, K, H, W = logits.shape
        hms.append(logits.flatten(2).softmax(-1).view(B, K, H, W).cpu())
    return torch.cat(uvs).numpy(), torch.cat(hms).numpy()


def to_native(uv224, meta):
    return uv224 * [meta["width"] / tc.IMG, meta["height"] / tc.IMG]


def to_224(uv_native, meta):
    return np.asarray(uv_native) * [tc.IMG / meta["width"], tc.IMG / meta["height"]]


# ------------------------------------------------------------ metrics over a whole split
def metrics_m(model, ds, samples, eps, device):
    """Ground-plane errors in meters: prediction vs actual path, and both vs the map centerline."""
    idx = list(range(len(samples)))
    pred224, _ = predict(model, ds, idx, device)
    e_gt, d_pred_c, d_gt_c = [], [], []
    for k, s in enumerate(samples):
        ctx = eps.context(s)
        pred_car = pixels_to_car(to_native(pred224[k], ctx["meta"]), ctx["pose"], ctx["meta"])
        gt_car = np.asarray(s["ego_xy"])
        cl = centerline_ahead(ctx, 0.0, 5.0)
        e_gt.append(np.linalg.norm(pred_car - gt_car, axis=1))
        d_pred_c.append(dist_to_polyline(pred_car, cl))
        d_gt_c.append(dist_to_polyline(gt_car, cl))
    e_gt, d_pred_c, d_gt_c = map(np.array, (e_gt, d_pred_c, d_gt_c))
    return {
        "pred_vs_actual_m (mean)": np.nanmean(e_gt, 0),
        "pred_vs_actual_m (median)": np.nanmedian(e_gt, 0),
        "pred_off_centerline_m (mean)": np.nanmean(d_pred_c, 0),
        "pred_off_centerline_m (median)": np.nanmedian(d_pred_c, 0),
        "actual_off_centerline_m (mean)": np.nanmean(d_gt_c, 0),
        "worst_frames": list(np.argsort(-np.nan_to_num(e_gt.mean(1), nan=99))[:6]),
        "n": len(samples),
    }


# ------------------------------------------------------------ visualisation
def show_panels(model, ds, samples, eps, device, picks, save=None):
    """One row per sample: heatmap overlay | paths in image | bird's-eye (meters)."""
    import matplotlib.pyplot as plt
    pred224, hms = predict(model, ds, picks, device)
    fig, axes = plt.subplots(len(picks), 3, figsize=(15, 4.6 * len(picks)))
    axes = np.atleast_2d(axes)
    for r, i in enumerate(picks):
        s, ctx = samples[i], eps.context(samples[i])
        meta, pose = ctx["meta"], ctx["pose"]
        img = ds.imgs[i]
        gt224 = np.asarray(ds.uv[i]); gv = np.asarray(ds.valid[i])
        cl_car = centerline_ahead(ctx)
        cl_uv, cl_vis = car_to_pixels(cl_car, pose, meta)
        cl224 = to_224(cl_uv, meta)
        pred_car = pixels_to_car(to_native(pred224[r], meta), pose, meta)
        gt_car = np.asarray(s["ego_xy"])

        # (1) heatmaps: max over the 6 waypoints, upsampled onto the image
        ax = axes[r, 0]
        hm = torch.from_numpy(hms[r].max(0))[None, None]
        hm = F.interpolate(hm, size=(tc.IMG, tc.IMG), mode="bilinear", align_corners=False)[0, 0].numpy()
        ax.imshow(img)
        ax.imshow(hm / hm.max(), cmap="inferno", alpha=0.55)
        ax.plot(gt224[gv, 0], gt224[gv, 1], "o", mfc="none", mec="lime", ms=8, mew=2, label="actual waypoints")
        ax.set_title("model heatmaps (all 6 waypoints)"); ax.axis("off"); ax.legend(loc="upper left", fontsize=8)

        # (2) paths in the image
        ax = axes[r, 1]
        ax.imshow(img)
        ax.plot(cl224[cl_vis, 0], cl224[cl_vis, 1], "-", color="deepskyblue", lw=3, alpha=0.9, label="map centerline (hard-coded)")
        ax.plot(gt224[gv, 0], gt224[gv, 1], "o-", color="lime", lw=2, ms=4, label="actual path driven")
        ax.plot(pred224[r][:, 0], pred224[r][:, 1], "o-", color="red", lw=2, ms=4, label="model prediction")
        ax.set_xlim(0, tc.IMG); ax.set_ylim(tc.IMG, 0); ax.axis("off")
        ax.set_title("paths in camera view"); ax.legend(loc="upper left", fontsize=8)

        # (3) bird's-eye in meters (car at origin, forward = up, left = left)
        ax = axes[r, 2]
        wx, wy = ctx["field"].walls_near(pose["x"], pose["y"])
        w_car = world_to_car(np.stack([wx, wy], 1), pose) if len(wx) else np.zeros((0, 2))
        ax.scatter(-w_car[:, 1], w_car[:, 0], s=1, c="0.6", label="ducts (map)")
        ax.plot(-cl_car[:, 1], cl_car[:, 0], "-", color="deepskyblue", lw=3, label="map centerline")
        ax.plot(-gt_car[:, 1], gt_car[:, 0], "o-", color="lime", lw=2, ms=4, label="actual path")
        ax.plot(-pred_car[:, 1], pred_car[:, 0], "o-", color="red", lw=2, ms=4, label="prediction")
        ax.plot([0], [0], marker=(3, 0, 0), color="k", ms=12)
        e = np.nanmean(np.linalg.norm(pred_car - gt_car, axis=1))
        dc = np.nanmean(dist_to_polyline(pred_car, cl_car))
        ax.set_title(f"top-down (m): err vs actual {e:.2f} m, off centerline {dc:.2f} m")
        ax.set_xlim(-3, 3); ax.set_ylim(-0.5, 4.5); ax.set_aspect("equal")
        ax.set_xlabel("← left   (m)   right →"); ax.set_ylabel("forward (m)")
        ax.grid(alpha=0.3); ax.legend(loc="lower right", fontsize=7)
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=100)
    plt.show()


def show_waypoint_heatmaps(model, ds, i, device):
    """The 6 individual heatmaps for one frame."""
    import matplotlib.pyplot as plt
    _, hms = predict(model, ds, [i], device)
    gt = np.asarray(ds.uv[i]); gv = np.asarray(ds.valid[i])
    fig, axes = plt.subplots(1, hms.shape[1], figsize=(3 * hms.shape[1], 3.2))
    for k, ax in enumerate(axes):
        h = F.interpolate(torch.from_numpy(hms[0, k])[None, None], size=(tc.IMG, tc.IMG),
                          mode="bilinear", align_corners=False)[0, 0].numpy()
        ax.imshow(ds.imgs[i]); ax.imshow(h / h.max(), cmap="inferno", alpha=0.6)
        if gv[k]:
            ax.plot(gt[k, 0], gt[k, 1], "o", mfc="none", mec="lime", ms=10, mew=2)
        ax.set_title(f"waypoint {k + 1}"); ax.axis("off")
    plt.tight_layout(); plt.show()


In [ ]:
# 2. Load model + test map
import sys, importlib, numpy as np, torch, pandas as pd
sys.path.insert(0, '/content')
import trace_core as tc, trace_ref as tr
importlib.reload(tc); importlib.reload(tr)

MAP = 'lab_15'        # unseen test map; set to e.g. 'lab_00' to look at a training map
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = tc.ViTHeatmap(n_wp=6, pretrained=False).to(device)
model.load_state_dict(torch.load(CKPT, map_location=device)); model.eval()

samples = tc.load_split('data', [MAP])
ds = tc.TrajDataset(samples)
eps = tr.Episodes(MAPS_DIR)
print(f'{MAP}: {len(samples)} frames | device {device}')

In [ ]:
# 3. Errors in METERS on the floor, per waypoint
m = tr.metrics_m(model, ds, samples, eps, device)
dist = [1.2 + 0.45 * k for k in range(6)]
table = pd.DataFrame({k: np.round(v, 3) for k, v in m.items() if k not in ('worst_frames', 'n')},
                     index=[f'wp{k+1} ({d:.2f} m ahead)' for k, d in enumerate(dist)])
display(table)
print('Columns: pred_vs_actual = model vs where the car really went;')
print('      pred_off_centerline = model vs the hard-coded map centerline;')
print('      actual_off_centerline = how far the autopilot itself was from the centerline (its own tracking error).')
WORST = m['worst_frames']

In [ ]:
# 4. Side-by-side panels: a few random frames + the worst ones
import random
random.seed(0)
picks = random.sample(range(len(samples)), 4)
tr.show_panels(model, ds, samples, eps, device, picks, save=f'{OUT}/inspect_random_{MAP}.png')

In [ ]:
# 5. The worst frames (largest error vs the actual path)
tr.show_panels(model, ds, samples, eps, device, WORST[:4], save=f'{OUT}/inspect_worst_{MAP}.png')

In [ ]:
# 6. The 6 individual waypoint heatmaps for one frame (change FRAME to any index)
FRAME = picks[0]
tr.show_waypoint_heatmaps(model, ds, FRAME, device)

### How to read it
- **Heatmaps**: a tight bright blob on each green circle means the model is confident and correct. A smeared or split blob means it's unsure, typically at hairpins or where the corridor forks.
- **Red vs green**: the model's error against what the expert actually did. This is what it was trained to match.
- **Red vs blue**: the model's error against the ideal reference line. Compare it with `actual_off_centerline` in the table: the model can't be expected to beat the autopilot it learned from.
- **Top-down view**: shows the error in real distances and how it compares to the ~1 m corridor. Red points staying between the grey ducts is what matters for driving.